# Eloundou_New — Exposure Score Rebuild

Merges new GPT-4 task labels from `Task_Statements_Classified_04.xlsx` into the Eloundou task-level and occupation-level files, recomputes α / β / γ scores, and produces diagnostics comparing old vs. new.

| File | Role |
|------|------|
| `full_labelset.tsv` | Eloundou task-level file (source of truth for task/occ structure) |
| `occ_level.csv` | Eloundou occupation-level scores (baseline) |
| `Task_Statements_Classified_04.xlsx` | New task labels (`exposure_label` column) |

## 1 · Imports & Config

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# ── Paths ────────────────────────────────────────────────────────────
BASE         = Path("/Users/nicobagnoli/Documents/PYTHON/SMP500-1")
UNTOUCHED    = BASE / "Eloundou_Untouched"
NEW_DIR      = BASE / "Eloundou_New"
NEW_DIR.mkdir(parents=True, exist_ok=True)

TASK_FILE    = UNTOUCHED / "full_labelset.tsv"
OCC_FILE     = UNTOUCHED / "occ_level.csv"
EXCEL_FILE   = BASE / "data" / "Task_Statements_Classified_04.xlsx"

OUT_TASK     = NEW_DIR / "full_labelset_new.tsv"
OUT_OCC      = NEW_DIR / "occ_level_new.csv"

# Supplemental tasks receive twice the weight of Core tasks
SUPPL_WEIGHT = 2.0
CORE_WEIGHT  = 1.0

print("Output directory:", NEW_DIR)
print("All input files exist:",
      all(p.exists() for p in [TASK_FILE, OCC_FILE, EXCEL_FILE]))

Output directory: /Users/nicobagnoli/Documents/PYTHON/SMP500-1/Eloundou_New
All input files exist: False


## 2 · Helper utilities

In [2]:
# ── Column auto-detection ────────────────────────────────────────────

def find_col(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    """Return the first column name (case-insensitive) that matches any candidate."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise KeyError(
        f"[auto-detect] Could not find '{label}' column. "
        f"Tried: {candidates}. Available: {list(df.columns)}"
    )


def detect_task_id(df: pd.DataFrame) -> str:
    return find_col(df, ["Task ID", "task_id", "taskid", "task id"], "Task ID")


def detect_occ_id(df: pd.DataFrame) -> str:
    return find_col(df, ["O*NET-SOC Code", "onet_soc_code", "soc_code",
                         "occupation_code", "occ_id", "O*NET SOC Code"], "Occupation ID")


def detect_task_type(df: pd.DataFrame) -> str:
    """Find the column that distinguishes Core vs. Supplemental tasks."""
    for col in df.columns:
        vals = df[col].dropna().astype(str).str.lower().unique()
        if any(v in vals for v in ["core", "supplemental", "supp"]):
            return col
    raise KeyError(
        "[auto-detect] Could not find a Core/Supplemental task-type column. "
        f"Available: {list(df.columns)}"
    )


# ── Eloundou scoring functions ───────────────────────────────────────

def to_bucket(label: str) -> str:
    """Map E0/E1/E2/E3 → E0, E1, E2  (E3 bucketed into E2)."""
    s = str(label).strip().upper()
    if s == "E3":
        return "E2"
    if s in {"E0", "E1", "E2"}:
        return s
    return "E0"   # treat unknown / NaN as E0


def alpha_score(bucket: str) -> float:
    """α = 1 iff task is E1."""
    return 1.0 if bucket == "E1" else 0.0


def beta_score(bucket: str) -> float:
    """β = 1 (E1), 0.5 (E2), 0 (E0)."""
    return {"E1": 1.0, "E2": 0.5, "E0": 0.0}.get(bucket, 0.0)


def gamma_score(bucket: str) -> float:
    """γ = 1 iff task is E1 or E2."""
    return 1.0 if bucket in {"E1", "E2"} else 0.0


print("Helpers defined.")

Helpers defined.


## 3 · Load inputs

In [3]:
df_tasks  = pd.read_csv(TASK_FILE, sep="\t", low_memory=False)
df_occ    = pd.read_csv(OCC_FILE)
df_excel  = pd.read_excel(EXCEL_FILE)

# Auto-detect key columns
TASK_COL       = detect_task_id(df_tasks)
TASK_COL_XL    = detect_task_id(df_excel)
OCC_COL_TASK   = detect_occ_id(df_tasks)
OCC_COL_OCC    = detect_occ_id(df_occ)
TYPE_COL       = detect_task_type(df_tasks)

print(f"full_labelset  : {len(df_tasks):,} rows  | task_id='{TASK_COL}' | occ_id='{OCC_COL_TASK}' | type='{TYPE_COL}'")
print(f"occ_level      : {len(df_occ):,} rows   | occ_id='{OCC_COL_OCC}'")
print(f"Excel classified: {len(df_excel):,} rows | task_id='{TASK_COL_XL}' | label col='exposure_label'")
print(f"\nExcel exposure_label distribution:")
print(df_excel["exposure_label"].value_counts().sort_index())

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nicobagnoli/Documents/PYTHON/SMP500-1/data/Task_Statements_Classified_04.xlsx'

## 4 · Build task-level file

In [ ]:
# ── Prepare Excel lookup ─────────────────────────────────────────────
xl_lookup = (
    df_excel[[TASK_COL_XL, "exposure_label"]]
    .drop_duplicates(subset=TASK_COL_XL, keep="last")
    .rename(columns={TASK_COL_XL: TASK_COL, "exposure_label": "new_labels"})
)

def normalise_task_id(series: pd.Series) -> pd.Series:
    """Convert float-like (8823.0) or int Task IDs to plain integer strings ('8823')."""
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype(str)

xl_lookup[TASK_COL] = normalise_task_id(xl_lookup[TASK_COL])
df_tasks[TASK_COL]  = normalise_task_id(df_tasks[TASK_COL])

# ── Merge new labels in ──────────────────────────────────────────────
df_new = df_tasks.merge(xl_lookup, on=TASK_COL, how="left")

n_matched   = df_new["new_labels"].notna().sum()
n_total     = len(df_new)
match_rate  = 100 * n_matched / n_total
print(f"Task match rate: {n_matched:,} / {n_total:,}  ({match_rate:.1f}%)")

if match_rate < 50:
    raise RuntimeError(
        f"Match rate too low ({match_rate:.1f}%). "
        "Check that Task ID columns reference the same IDs in both files."
    )

# ── Compute new scores ───────────────────────────────────────────────
df_new["_bucket_new"] = df_new["new_labels"].apply(to_bucket)
df_new["alpha_new"]   = df_new["_bucket_new"].apply(alpha_score)
df_new["beta_new"]    = df_new["_bucket_new"].apply(beta_score)
df_new["gamma_new"]   = df_new["_bucket_new"].apply(gamma_score)

# Drop helper column
df_new.drop(columns=["_bucket_new"], inplace=True)

# ── Save ─────────────────────────────────────────────────────────────
df_new.to_csv(OUT_TASK, sep="\t", index=False)
print(f"\nSaved: {OUT_TASK}  ({len(df_new):,} rows)")
df_new[[TASK_COL, "new_labels", "alpha_new", "beta_new", "gamma_new"]].head()


Task match rate: 18,729 / 19,265  (97.2%)

Saved: /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/full_labelset_new.tsv  (19,265 rows)


,Task ID,new_labels,alpha_new,beta_new,gamma_new
0,8823,E2,0.0,0.5,1.0
1,8831,E0,0.0,0.0,0.0
2,8825,E2,0.0,0.5,1.0
3,8826,E2,0.0,0.5,1.0
4,8827,E2,0.0,0.5,1.0


## 5 · Build occupation-level file

In [ ]:
# ── Assign task weights  (supplemental = 2 × core) ──────────────────
type_vals = df_new[TYPE_COL].astype(str).str.lower().str.strip()
df_new["_weight"] = type_vals.apply(
    lambda t: SUPPL_WEIGHT if "suppl" in t else CORE_WEIGHT
)

# ── Weighted aggregation per occupation ─────────────────────────────
def weighted_mean(group: pd.DataFrame, score_col: str) -> float:
    w = group["_weight"]
    s = group[score_col]
    # Only include rows where the new label is available
    mask = s.notna()
    w, s = w[mask], s[mask]
    if w.sum() == 0:
        return np.nan
    return (w * s).sum() / w.sum()

records = []
for occ_id, grp in df_new.groupby(OCC_COL_TASK):
    records.append({
        OCC_COL_OCC: occ_id,
        "alpha_new": weighted_mean(grp, "alpha_new"),
        "beta_new":  weighted_mean(grp, "beta_new"),
        "gamma_new": weighted_mean(grp, "gamma_new"),
    })

df_occ_new_scores = pd.DataFrame(records)
df_occ_new_scores["dv_rating_new"] = df_occ_new_scores["beta_new"]

# ── Merge into original occ_level ────────────────────────────────────
df_occ[OCC_COL_OCC] = df_occ[OCC_COL_OCC].astype(str).str.strip()
df_occ_new_scores[OCC_COL_OCC] = df_occ_new_scores[OCC_COL_OCC].astype(str).str.strip()

df_occ_out = df_occ.merge(df_occ_new_scores, on=OCC_COL_OCC, how="left")

n_occ_matched = df_occ_out["beta_new"].notna().sum()
print(f"Occupations with new scores: {n_occ_matched:,} / {len(df_occ_out):,}")

# ── Save ─────────────────────────────────────────────────────────────
df_occ_out.to_csv(OUT_OCC, index=False)
print(f"Saved: {OUT_OCC}  ({len(df_occ_out):,} rows)")
df_occ_out.head()

Occupations with new scores: 923 / 923
Saved: /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/occ_level_new.csv  (923 rows)


,O*NET-SOC Code,Title,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma,alpha_new,beta_new,gamma_new,dv_rating_new
0,11-1011.00,Chief Executives,0.100000,0.460000,0.820000,0.180000,0.350000,0.520000,0.279070,0.430233,0.581395,0.430233
1,11-1011.03,Chief Sustainability Officers,0.166667,0.555556,0.944444,0.055556,0.388889,0.722222,0.666667,0.805556,0.944444,0.805556
2,11-1021.00,General and Operations Managers,0.000000,0.480769,0.961538,0.115385,0.384615,0.653846,0.200000,0.560000,0.920000,0.560000
3,11-1031.00,Legislators,0.033333,0.400000,0.766667,0.266667,0.516667,0.766667,0.366667,0.366667,0.366667,0.366667
4,11-2011.00,Advertising and Promotions Managers,0.000000,0.476744,0.953488,0.255814,0.546512,0.837209,0.552632,0.605263,0.657895,0.605263


## 6 · Diagnostics

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 6-A  Task-label distribution  (original GPT-4 vs. new)
# ──────────────────────────────────────────────────────────────────────

# Detect original GPT-4 label column
orig_label_col = find_col(
    df_new,
    ["gpt4_exposure", "gpt4_label", "gpt_4_exposure", "gpt4"],
    "original GPT-4 label"
)

df_new["_bucket_orig"] = df_new[orig_label_col].apply(to_bucket)
df_new["_bucket_new2"] = df_new["new_labels"].apply(to_bucket)

BUCKETS = ["E0", "E1", "E2"]
N = len(df_new)

orig_dist = df_new["_bucket_orig"].value_counts().reindex(BUCKETS, fill_value=0)
new_dist  = df_new["_bucket_new2"].value_counts().reindex(BUCKETS, fill_value=0)

print("=" * 60)
print("TASK LABEL DISTRIBUTION  (E3 → E2)")
print(f"{'Bucket':>10}  {'Orig GPT-4':>12}  {'New Labels':>12}")
print("-" * 40)
for b in BUCKETS:
    o, nv = orig_dist[b], new_dist[b]
    print(f"{b:>10}  {o:>6,} ({100*o/N:5.1f}%)  {nv:>6,} ({100*nv/N:5.1f}%)")
print("=" * 60)

TASK LABEL DISTRIBUTION  (E3 → E2)
    Bucket    Orig GPT-4    New Labels
----------------------------------------
        E0   8,561 ( 44.4%)  10,742 ( 55.8%)
        E1   2,702 ( 14.0%)   4,677 ( 24.3%)
        E2   8,002 ( 41.5%)   3,846 ( 20.0%)


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 6-B  Transition / flow matrix  (orig → new)
# ──────────────────────────────────────────────────────────────────────

flow = pd.crosstab(
    df_new["_bucket_orig"].rename("Original"),
    df_new["_bucket_new2"].rename("New"),
    margins=True
).reindex(index=BUCKETS + ["All"], columns=BUCKETS + ["All"], fill_value=0)

print("\nCOUNT matrix  (rows = original, cols = new):")
print(flow.to_string())

# Row-normalised (percent of original bucket)
flow_pct = (
    flow.loc[BUCKETS, BUCKETS]
    .div(flow.loc[BUCKETS, "All"], axis=0)
    .mul(100)
    .round(1)
)
print("\nROW-NORMALISED matrix  (% of each original bucket):")
print(flow_pct.to_string())


COUNT matrix  (rows = original, cols = new):
New          E0    E1    E2    All
Original                          
E0         8125   189   247   8561
E1          333  1567   802   2702
E2         2284  2921  2797   8002
All       10742  4677  3846  19265

ROW-NORMALISED matrix  (% of each original bucket):
New         E0    E1    E2
Original                  
E0        94.9   2.2   2.9
E1        12.3  58.0  29.7
E2        28.5  36.5  35.0


Interpretation

2284 tasks that were originally classified as E2 are now classified as E0.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 6-C  Occupation beta scores  (original vs. new)
# ──────────────────────────────────────────────────────────────────────

# Auto-detect original beta column in occ_level
beta_orig_col = find_col(
    df_occ_out,
    ["dv_rating_beta", "beta", "dv_rating", "dv_rating_b"],
    "original occupation beta"
)

valid = df_occ_out[[OCC_COL_OCC, "Title", beta_orig_col, "beta_new"]].dropna()

r, p = pearsonr(valid[beta_orig_col], valid["beta_new"])

print("=" * 60)
print(f"OCCUPATION BETA CORRELATION")
print(f"  Original col   : '{beta_orig_col}'")
print(f"  Occupations    : {len(valid):,}")
print(f"  Pearson r      : {r:.4f}")
print(f"  p-value        : {p:.2e}")
print("=" * 60)

# ── Top 10 by absolute difference ───────────────────────────────────
valid = valid.copy()
valid["diff"] = valid["beta_new"] - valid[beta_orig_col]
valid["abs_diff"] = valid["diff"].abs()

top10 = valid.nlargest(10, "abs_diff")[["Title", beta_orig_col, "beta_new", "diff"]]
top10.columns = ["Occupation", "beta_orig", "beta_new", "diff"]

print("\nTOP 10 OCCUPATIONS BY |β difference|:")
print(top10.to_string(index=False))

OCCUPATION BETA CORRELATION
  Original col   : 'dv_rating_beta'
  Occupations    : 923
  Pearson r      : 0.8761
  p-value        : 5.39e-294

TOP 10 OCCUPATIONS BY |β difference|:
                                 Occupation  beta_orig  beta_new      diff
                                 Physicists   0.750000  0.347826 -0.402174
                        Telephone Operators   0.894737  0.500000 -0.394737
                      Correspondence Clerks   0.964286  0.571429 -0.392857
             Payroll and Timekeeping Clerks   0.837838  0.448276 -0.389562
                              Tax Preparers   0.625000  1.000000  0.375000
Court Reporters and Simultaneous Captioners   0.958333  0.583333 -0.375000
            Paralegals and Legal Assistants   0.525000  0.156250 -0.368750
                 Financial Risk Specialists   0.533333  0.883333  0.350000
                   Environmental Economists   0.605263  0.954545  0.349282
             Subway and Streetcar Operators   0.409091  0.071429 -0.3

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 6-D  Short recap
# ──────────────────────────────────────────────────────────────────────

print("\n" + "━" * 60)
print("RECAP")
print("━" * 60)
print(f"  Task match rate        : {match_rate:.1f}%")
print(f"  Tasks with new labels  : {n_matched:,} / {n_total:,}")
print()
print("  Label distribution shift (E0 / E1 / E2):")
for b in BUCKETS:
    o, nv = orig_dist[b], new_dist[b]
    delta = nv - o
    sign  = "+" if delta >= 0 else ""
    print(f"    {b}: {100*o/N:.1f}% → {100*nv/N:.1f}%  ({sign}{delta:,} tasks)")
print()
print(f"  Occupation β Pearson r : {r:.4f}  (p={p:.2e})")
print(f"  Output task file       : {OUT_TASK}")
print(f"  Output occ  file       : {OUT_OCC}")
print("━" * 60)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RECAP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Task match rate        : 97.2%
  Tasks with new labels  : 18,729 / 19,265

  Label distribution shift (E0 / E1 / E2):
    E0: 44.4% → 55.8%  (+2,181 tasks)
    E1: 14.0% → 24.3%  (+1,975 tasks)
    E2: 41.5% → 20.0%  (-4,156 tasks)

  Occupation β Pearson r : 0.8761  (p=5.39e-294)
  Output task file       : /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/full_labelset_new.tsv
  Output occ  file       : /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/occ_level_new.csv
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 7 · Time-based occupation score

Computes a time-weighted average of the Eloundou β task score using task labor shares (π) from the ONET standard task-labor-share file.

$$\text{time\_based\_score}_o = \frac{\sum_{t \in T_o} \beta_t \cdot \pi_t}{\sum_{t \in T_o} \pi_t}$$

Occupations for which **any** task is missing a valid π are excluded (NaN).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 7 · Time-based occupation score
#     time_based_score_o = Σ(β_t * π_t) / Σ(π_t)   for all t in occ o
#     Occupations with ANY missing π → NaN (excluded entirely)
# ══════════════════════════════════════════════════════════════════════

TIME_FILE = BASE / "Time" / "New_pi.csv"

# ── 1. Load files ─────────────────────────────────────────────────────
df_ts_new  = pd.read_csv(OUT_TASK, sep="\t", low_memory=False)   # full_labelset_new.tsv
df_occ_out = pd.read_csv(OUT_OCC)                                 # occ_level_new.csv
df_time    = pd.read_csv(TIME_FILE)

print(f"full_labelset_new rows : {len(df_ts_new):,}")
print(f"occ_level_new rows     : {len(df_occ_out):,}")
print(f"time-share rows (raw)  : {len(df_time):,}")

# ── 2. Auto-detect columns ───────────────────────────────────────────
#  Task-level file: task ID, occupation ID
TID_COL  = detect_task_id(df_ts_new)     # 'Task ID'
TOCC_COL = detect_occ_id(df_ts_new)      # 'O*NET-SOC Code'

#  occ_level_new: occupation ID column (detected independently)
OCC_OUT_COL = detect_occ_id(df_occ_out)  # 'O*NET-SOC Code'

#  Require beta_new explicitly — fail loudly if absent
if "beta_new" not in df_ts_new.columns:
    raise RuntimeError(
        "FATAL: 'beta_new' column not found in full_labelset_new.tsv. "
        "Cannot compute time_based_score without the Eloundou beta_new task score."
    )

#  Time-share file – detect task_id, occ_id, and pi columns
def detect_time_task_id(df):
    return find_col(df, ["task_id", "Task ID", "taskid", "task id"], "task_id (time-share)")

def detect_time_occ_id(df):
    return find_col(df, ["onetsoc_code", "O*NET-SOC Code", "onet_soc_code",
                         "soc_code", "occupation_code"], "occ_id (time-share)")

def detect_pi_col(df):
    return find_col(df, ["pi", "labor_share", "task_labor_share", "time_share",
                         "weight", "pi_t"], "pi (labor share)")

TT_ID_COL  = detect_time_task_id(df_time)    # 'task_id'
TT_OCC_COL = detect_time_occ_id(df_time)     # 'onetsoc_code'
PI_COL     = detect_pi_col(df_time)          # 'pi'

print(f"\nTime-share columns detected: task='{TT_ID_COL}', occ='{TT_OCC_COL}', pi='{PI_COL}'")

# ── 3. Normalise task IDs & occupation codes ─────────────────────────
def norm_task_id(s):
    """Float-like '8823.0' or int → plain integer string '8823'."""
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype(str)

def norm_occ_id(s):
    return s.astype(str).str.strip()

df_ts_new[TID_COL]   = norm_task_id(df_ts_new[TID_COL])
df_ts_new[TOCC_COL]  = norm_occ_id(df_ts_new[TOCC_COL])

df_time[TT_ID_COL]   = norm_task_id(df_time[TT_ID_COL])
df_time[TT_OCC_COL]  = norm_occ_id(df_time[TT_OCC_COL])

df_occ_out[OCC_OUT_COL] = norm_occ_id(df_occ_out[OCC_OUT_COL])

# ── 4. Check for duplicate (occ, task) pairs in time-share file ──────
dup_mask = df_time.duplicated(subset=[TT_OCC_COL, TT_ID_COL], keep=False)
n_dup    = dup_mask.sum()

if n_dup > 0:
    raise RuntimeError(
        f"FATAL: {n_dup:,} duplicate (occupation, task) rows found in time-share file "
        f"'{TIME_FILE.name}'. Cannot aggregate ambiguously — please inspect before proceeding."
    )
print(f"Duplicate (occ, task) pairs in time-share file: {n_dup}  OK")

# ── 5. Build clean pi lookup: (occ_id, task_id) → pi ─────────────────
df_pi = (
    df_time[[TT_OCC_COL, TT_ID_COL, PI_COL]]
    .rename(columns={TT_OCC_COL: "occ_key", TT_ID_COL: "task_key", PI_COL: "pi"})
)

# ── 6. Extract task-level beta_new with occ + task IDs ───────────────
df_tasks_sub = (
    df_ts_new[[TOCC_COL, TID_COL, "beta_new"]]
    .rename(columns={TOCC_COL: "occ_key", TID_COL: "task_key"})
)

print(f"\nTask rows in full_labelset_new : {len(df_tasks_sub):,}")

# ── 7. Merge beta_new with pi on (occ_id, task_id) ───────────────────
df_merged = df_tasks_sub.merge(
    df_pi, on=["occ_key", "task_key"], how="left"
)

n_pi_matched = df_merged["pi"].notna().sum()
n_pi_total   = len(df_merged)
print(f"Task rows matched to pi        : {n_pi_matched:,} / {n_pi_total:,}"
      f"  ({100*n_pi_matched/n_pi_total:.1f}%)")
print("(Merge key: occupation ID + task ID)")

# ── 8. Compute time_based_score per occupation ────────────────────────
#  Rule: if ANY task in an occupation is missing pi → exclude that occ
records_time = []
ignored_occs = []
scored_occs  = []

for occ_id, grp in df_merged.groupby("occ_key"):
    n_tasks   = len(grp)
    n_missing = grp["pi"].isna().sum()

    if n_missing > 0:
        ignored_occs.append({"occ_key": occ_id, "n_tasks": n_tasks,
                              "n_missing_pi": n_missing})
        records_time.append({"occ_key": occ_id, "time_based_score": np.nan})
    else:
        pi_vals   = grp["pi"].values.astype(float)
        beta_vals = grp["beta_new"].values.astype(float)
        valid     = ~np.isnan(beta_vals)
        pi_v, beta_v = pi_vals[valid], beta_vals[valid]

        if pi_v.sum() == 0 or len(pi_v) == 0:
            ignored_occs.append({"occ_key": occ_id, "n_tasks": n_tasks,
                                  "n_missing_pi": 0})
            records_time.append({"occ_key": occ_id, "time_based_score": np.nan})
        else:
            score = float(np.dot(beta_v, pi_v) / pi_v.sum())
            scored_occs.append({"occ_key": occ_id, "time_based_score": score})
            records_time.append({"occ_key": occ_id, "time_based_score": score})

df_time_scores = pd.DataFrame(records_time)

n_scored  = len(scored_occs)
n_ignored = len(ignored_occs)
n_total   = n_scored + n_ignored
pct_ign   = 100 * n_ignored / n_total if n_total > 0 else 0.0

# ── 9. Merge into occ_level_new and write ────────────────────────────
df_time_scores = df_time_scores.rename(columns={"occ_key": OCC_OUT_COL})

# Drop existing time_based_score column if present (overwrite)
if "time_based_score" in df_occ_out.columns:
    df_occ_out = df_occ_out.drop(columns=["time_based_score"])

df_occ_out = df_occ_out.merge(df_time_scores, on=OCC_OUT_COL, how="left")
df_occ_out.to_csv(OUT_OCC, index=False)

# ── 10. Diagnostics ──────────────────────────────────────────────────
print("\n" + "=" * 64)
print("TIME-BASED OCCUPATION SCORE  --  DIAGNOSTICS")
print("=" * 64)
print(f"  Task rows in full_labelset_new.tsv        : {n_pi_total:,}")
print(f"  Task rows matched to time-share pi         : {n_pi_matched:,}")
print(f"  Occupations successfully scored            : {n_scored:,} / {n_total:,}")
print(f"  Occupations ignored (incomplete pi)        : {n_ignored:,} / {n_total:,}")
print(f"  Percentage ignored                         : {pct_ign:.1f}%")

# Preview scored
df_scored_preview = pd.DataFrame(scored_occs).head(8)
if len(df_scored_preview) > 0:
    print(f"\n  Preview -- occupations with valid time_based_score ({min(8, n_scored)} shown):")
    print(df_scored_preview.to_string(index=False))

# Preview excluded
df_ignored_preview = pd.DataFrame(ignored_occs).head(8)
if len(df_ignored_preview) > 0:
    print(f"\n  Preview -- occupations excluded due to missing pi ({min(8, n_ignored)} shown):")
    print(df_ignored_preview.to_string(index=False))

# ── 11. Correlation: beta_new (occ-level) vs time_based_score ────────
both_valid = df_occ_out[["beta_new", "time_based_score"]].dropna()
if len(both_valid) >= 2:
    r_tb, p_tb = pearsonr(both_valid["beta_new"], both_valid["time_based_score"])
    print(f"\n  Pearson r (beta_new vs time_based_score)  : {r_tb:.4f}  (p={p_tb:.2e})")
    print(f"  Based on {len(both_valid):,} occupations with both values available")
else:
    print("\n  Not enough overlapping occupations to compute correlation.")

print("\n" + "-" * 64)
print("SUMMARY")
print("-" * 64)
print(f"  time_based_score written to {OUT_OCC}")
print(f"  occupations scored               : {n_scored} / {n_total}")
print(f"  occupations ignored due to")
print(f"    incomplete pi coverage         : {n_ignored} / {n_total}")
print(f"  percent ignored                  : {pct_ign:.1f}%")
print("-" * 64)


full_labelset_new rows : 19,265
occ_level_new rows     : 923
time-share rows (raw)  : 16,597

Time-share columns detected: task='task_id', occ='onetsoc_code', pi='pi'
Duplicate (occ, task) pairs in time-share file: 0  OK

Task rows in full_labelset_new : 19,265
Task rows matched to pi        : 16,547 / 19,265  (85.9%)
(Merge key: occupation ID + task ID)

TIME-BASED OCCUPATION SCORE  --  DIAGNOSTICS
  Task rows in full_labelset_new.tsv        : 19,265
  Task rows matched to time-share pi         : 16,547
  Occupations successfully scored            : 332 / 923
  Occupations ignored (incomplete pi)        : 591 / 923
  Percentage ignored                         : 64.0%

  Preview -- occupations with valid time_based_score (8 shown):
   occ_key  time_based_score
11-1011.03          0.550623
11-2022.00          0.507958
11-3013.01          0.620478
11-3031.01          0.414790
11-3051.02          0.278809
11-3051.03          0.160477
11-3051.04          0.323614
11-3061.00          0.5639